# Xây dựng corpus điều luật

Đọc corpus VBPL đã lọc, khám phá metadata và tách HTML thành corpus mức Điều `results/articles.json`.

Thành phần:
- `DocumentCatalog` — metadata tổng hợp (lookup theo id, tìm theo từ khoá, thống kê).
- `VbplDocument` — một văn bản chi tiết (thuộc tính + HTML), sinh `header` (số hiệu + tên).
- `DocumentRepository` — đọc file chi tiết trong `filtered/json/` (fallback `data/`).
- `Article` / `ArticleParser` — tách `content_html` thành các Điều theo class `prov-*` (fallback regex).
- `ArticleCorpusBuilder` — gộp toàn bộ thành dict `{dieu_id: noi_dung}` và lưu file.

Output `results/articles.json`: dict `{"{doc_id}_{số điều}": noi_dung}`, mỗi `noi_dung` ghép số hiệu + tên văn bản lên đầu.

## 1. Cấu hình

In [1]:
from __future__ import annotations

import re
import os
import json
import glob
from collections import Counter
from dataclasses import dataclass

from bs4 import BeautifulSoup

FILTERED_JSON   = "../R2AI_VBPL/filtered/filtered_by_agency_type_active_after_20231128.json"
JSON_DETAIL_DIR = "../R2AI_VBPL/filtered/json"
DATA_DIR        = "../R2AI_VBPL/filtered/json"  # fallback = cùng thư mục (không có data/ riêng)
OUTPUT_PATH     = "../R2AI_VBPL/articles.json"

DOC_ID = "175155"  # id dùng để minh hoạ

## 2. Định nghĩa lớp

In [2]:
@dataclass
class Article:
    """Một điều luật tách từ HTML."""
    dieu_id: int | None   # số điều
    noi_dung: str         # toàn bộ nội dung điều (gồm dòng 'Điều N. ...')


class ArticleParser:
    """Tách content_html thành list[Article].

    Mỗi <p class='prov-article'> mở một Điều; các thẻ prov-* phía sau thuộc Điều đó
    cho tới prov-article kế tiếp. Văn bản không có prov-* dùng fallback regex.
    """

    @staticmethod
    def _norm(s: str) -> str:
        return re.sub(r"\s+", " ", (s or "").replace("\xa0", " ")).strip()

    def parse(self, content_html: str) -> list[Article]:
        soup = BeautifulSoup(content_html or "", "html.parser")
        blocks = []
        for p in soup.find_all("p"):
            prov = next((c for c in (p.get("class") or []) if c.startswith("prov-")), None)
            txt = self._norm(p.get_text())
            if prov and txt:
                blocks.append((prov, txt))

        if not any(c == "prov-article" for c, _ in blocks):
            return self._parse_fallback(content_html)

        articles, cur = [], None
        for prov, txt in blocks:
            if prov == "prov-article":
                if cur:
                    articles.append(cur)
                m = re.match(r"Điều\s+(\d+)", txt)
                cur = {"dieu_id": int(m.group(1)) if m else None, "lines": [txt]}
            elif cur is not None:
                cur["lines"].append(txt)
        if cur:
            articles.append(cur)
        return [Article(a["dieu_id"], "\n".join(a["lines"])) for a in articles]

    def _parse_fallback(self, content_html: str) -> list[Article]:
        text = self._norm(BeautifulSoup(content_html or "", "html.parser").get_text())
        out = []
        for seg in re.split(r"(?=Điều\s+\d+\s*[.:])", text):
            m = re.match(r"Điều\s+(\d+)", seg)
            if m:
                out.append(Article(int(m.group(1)), self._norm(seg)))
        return out


@dataclass
class VbplDocument:
    """Một văn bản chi tiết (file filtered/json/<id>.json)."""
    doc_id: str
    thuoc_tinh: dict
    content_html: str

    @classmethod
    def from_detail(cls, detail: dict) -> "VbplDocument":
        return cls(
            doc_id=str(detail.get("doc_id") or detail.get("thuoc_tinh", {}).get("id", "")),
            thuoc_tinh=detail.get("thuoc_tinh", {}),
            content_html=detail.get("noi_dung", {}).get("content_html", ""),
        )

    @property
    def so_hieu(self) -> str:
        return self.thuoc_tinh.get("so_hieu", "") or ""

    @property
    def ten_van_ban(self) -> str:
        return self.thuoc_tinh.get("ten_van_ban", "") or ""

    @property
    def header(self) -> str:
        return f"{self.so_hieu}\n{self.ten_van_ban}".strip()


class DocumentRepository:
    """Đọc file chi tiết theo id hoặc duyệt toàn bộ thư mục."""

    def __init__(self, json_dir: str, data_dir: str | None = None):
        self.json_dir = json_dir
        self.data_dir = data_dir

    def _path(self, doc_id: str) -> str | None:
        p = os.path.join(self.json_dir, f"{doc_id}.json")
        if os.path.exists(p):
            return p
        if self.data_dir:
            p = os.path.join(self.data_dir, f"{doc_id}.json")
            if os.path.exists(p):
                return p
        return None

    def load(self, doc_id: str) -> VbplDocument | None:
        path = self._path(doc_id)
        if not path:
            return None
        with open(path, encoding="utf-8") as f:
            return VbplDocument.from_detail(json.load(f))

    def iter_documents(self):
        for fn in glob.glob(os.path.join(self.json_dir, "*.json")):
            with open(fn, encoding="utf-8") as f:
                yield VbplDocument.from_detail(json.load(f))

    def __len__(self) -> int:
        return len(glob.glob(os.path.join(self.json_dir, "*.json")))


class ArticleCorpusBuilder:
    """Gộp các văn bản thành dict {dieu_id: noi_dung} và lưu file."""

    def __init__(self, repo: DocumentRepository, parser: ArticleParser):
        self.repo = repo
        self.parser = parser

    def build_for(self, doc: VbplDocument) -> dict[str, str]:
        """Một văn bản -> {f'{doc_id}_{số điều}': header + noi_dung}."""
        out: dict[str, str] = {}
        for a in self.parser.parse(doc.content_html):
            key = f"{doc.doc_id}_{a.dieu_id}"
            while key in out:           # giữ key duy nhất khi lặp số điều
                key += "_x"
            out[key] = f"{doc.header}\n{a.noi_dung}" if doc.header else a.noi_dung
        return out

    def build_all(self) -> tuple[dict[str, str], int, int]:
        corpus: dict[str, str] = {}
        n_files = n_empty = 0
        for doc in self.repo.iter_documents():
            arts = self.build_for(doc)
            n_files += 1
            if not arts:
                n_empty += 1
            corpus.update(arts)
        return corpus, n_files, n_empty

    @staticmethod
    def save(corpus: dict[str, str], path: str) -> None:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(corpus, f, ensure_ascii=False, indent=1)
        print(f"Đã lưu {len(corpus)} điều vào {path}")


class DocumentCatalog:
    """Metadata tổng hợp: lookup theo id, tìm theo từ khoá, thống kê."""

    def __init__(self, documents: list[dict]):
        self.documents = documents
        self._index = {d["id"]: d for d in documents}

    @classmethod
    def from_json(cls, path: str) -> "DocumentCatalog":
        with open(path, encoding="utf-8") as f:
            return cls(json.load(f))

    def get(self, doc_id: str) -> dict | None:
        return self._index.get(doc_id)

    def search(self, keyword: str) -> list[dict]:
        kw = keyword.lower()
        return [d for d in self.documents if kw in d.get("ten_van_ban", "").lower()]

    def stats(self) -> dict:
        return {
            "n_docs": len(self.documents),
            "by_type": Counter(d["loai_van_ban"] for d in self.documents),
            "by_agency": Counter(d["co_quan_ban_hanh"] for d in self.documents),
            "with_html": sum(1 for d in self.documents if d.get("content_length", 0) > 0),
        }

    def __len__(self) -> int:
        return len(self.documents)

## 3. Khám phá metadata

In [3]:
catalog = DocumentCatalog.from_json(FILTERED_JSON)
print(f"Catalog: {len(catalog)} văn bản")

st = catalog.stats()
print("\nTheo loại văn bản:")
for t, c in st["by_type"].most_common():
    print(f"  {t:25s}: {c:4d}")
print(f"\nCó nội dung HTML: {st['with_html']}/{st['n_docs']}")

doc_meta = catalog.get(DOC_ID)
print(f"\n[{DOC_ID}] {doc_meta['ten_van_ban'] if doc_meta else 'không tìm thấy'}")
print(f"Số văn bản chứa 'thuế': {len(catalog.search('thuế'))}")

Catalog: 5 văn bản

Theo loại văn bản:
  Thông tư                 :    5

Có nội dung HTML: 5/5

[175155] Thông tư số 10/2025/TT-BGTVT Ban hành Quy chuẩn kỹ thuật quốc gia về thiết bị nâng trên các công trình biển.
Số văn bản chứa 'thuế': 0


## 4. Đọc chi tiết và tách điều cho một văn bản

In [4]:
repo = DocumentRepository(JSON_DETAIL_DIR, DATA_DIR)
parser = ArticleParser()
builder = ArticleCorpusBuilder(repo, parser)

doc = repo.load(DOC_ID)
articles = builder.build_for(doc) if doc else {}
print(f"[{DOC_ID}] {doc.so_hieu} - tách được {len(articles)} điều\n")
for key, noi_dung in list(articles.items())[:3]:
    print(f"--- {key} ---")
    print(noi_dung[:300])
    print()

[175155] 10/2025/TT-BGTVT - tách được 3 điều

--- 175155_1 ---
10/2025/TT-BGTVT
Thông tư số 10/2025/TT-BGTVT Ban hành Quy chuẩn kỹ thuật quốc gia về thiết bị nâng trên các công trình biển.
Điều 1. Ban hành kèm theo Thông này Quy chuẩn kỹ thuật quốc gia về thiết bị nâng trên các công trình biển.
Mã số đăng ký: QCVN 97:2025/BGTVT.

--- 175155_2 ---
10/2025/TT-BGTVT
Thông tư số 10/2025/TT-BGTVT Ban hành Quy chuẩn kỹ thuật quốc gia về thiết bị nâng trên các công trình biển.
Điều 2. Hiệu lực thi hành
1. Thông tư này có hiệu lực kể từ ngày 20 tháng 8 năm 2025.
2. Giấy chứng nhận kết quả kiểm định cấp cho thiết bị nâng trước ngày Thông tư này có hi

--- 175155_3 ---
10/2025/TT-BGTVT
Thông tư số 10/2025/TT-BGTVT Ban hành Quy chuẩn kỹ thuật quốc gia về thiết bị nâng trên các công trình biển.
Điều 3. Tổ chức thực hiện
Chánh Văn phòng Bộ, Chánh Thanh tra Bộ, Vụ trưởng các Vụ, Cục trưởng Cục Đăng kiểm Việt Nam, Thủ trưởng các đơn vị thuộc Bộ Giao thông vận tải, các 



## 5. Tách toàn bộ và lưu corpus

In [5]:
corpus, n_files, n_empty = builder.build_all()
ArticleCorpusBuilder.save(corpus, OUTPUT_PATH)
print(f"{n_files} file -> {len(corpus)} điều")
print(f"{n_empty} file không tách được điều nào (thường là văn bản định mức/bảng biểu)")

Đã lưu 25 điều vào ../R2AI_VBPL/articles.json
5 file -> 25 điều
0 file không tách được điều nào (thường là văn bản định mức/bảng biểu)


## 6. Xem thử corpus

In [6]:
for key in list(corpus)[:2]:
    print(f"--- {key} ---")
    print(corpus[key][:300])
    print()

--- 163716_1 ---
12/2023/TT-BNNPTNT
Thông tư số 12/2023/TT-BNNPTNT Hướng dẫn về Hội đồng quản lý và tiêu chuẩn, điều kiện bổ nhiệm, miễn nhiệm thành viên Hội đồng quản lý trong đơn vị sự nghiệp công lập thuộc lĩnh vực nông nghiệp và phát triển nông thôn
Điều 1. Phạm vi điều chỉnh
Thông tư này hướng dẫn về chức năng

--- 163716_2 ---
12/2023/TT-BNNPTNT
Thông tư số 12/2023/TT-BNNPTNT Hướng dẫn về Hội đồng quản lý và tiêu chuẩn, điều kiện bổ nhiệm, miễn nhiệm thành viên Hội đồng quản lý trong đơn vị sự nghiệp công lập thuộc lĩnh vực nông nghiệp và phát triển nông thôn
Điều 2. Đối tượng áp dụng
1. Các đơn vị sự nghiệp công lập thuộ

